In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ====================================
# CELDA 1: Instalación simplificada
# ====================================
!pip install -q gradio==3.50.2

# ====================================
# CELDA 2: Importaciones
# ====================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from PIL import Image
import gradio as gr
import torch

# ====================================
# CELDA 3: Generador Perlin
# ====================================

def fade(t):
    """Función de suavizado para Perlin noise"""
    return 6*t**5 - 15*t**4 + 10*t**3

def perlin_noise(width, height, scale=12):
    """Genera ruido Perlin para texturas procedurales"""
    grid_w = width // scale + 2
    grid_h = height // scale + 2

    gradients = np.random.randn(grid_h, grid_w, 2)
    norms = np.linalg.norm(gradients, axis=2, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    gradients = gradients / norms

    xs = np.linspace(0, grid_w - 2, width)
    ys = np.linspace(0, grid_h - 2, height)
    x, y = np.meshgrid(xs, ys)

    x0 = np.clip(x.astype(int), 0, grid_w - 2)
    x1 = np.clip(x0 + 1, 0, grid_w - 1)
    y0 = np.clip(y.astype(int), 0, grid_h - 2)
    y1 = np.clip(y0 + 1, 0, grid_h - 1)

    dx = x - x0
    dy = y - y0

    g00 = gradients[y0, x0]
    g10 = gradients[y0, x1]
    g01 = gradients[y1, x0]
    g11 = gradients[y1, x1]

    dot00 = g00[:,:,0]*dx + g00[:,:,1]*dy
    dot10 = g10[:,:,0]*(dx-1) + g10[:,:,1]*dy
    dot01 = g01[:,:,0]*dx + g01[:,:,1]*(dy-1)
    dot11 = g11[:,:,0]*(dx-1) + g11[:,:,1]*(dy-1)

    u = fade(dx)
    v = fade(dy)

    noise = (1-u)*(1-v)*dot00 + u*(1-v)*dot10 + (1-u)*v*dot01 + u*v*dot11

    noise_min = noise.min()
    noise_max = noise.max()
    if noise_max - noise_min > 0:
        noise = (noise - noise_min) / (noise_max - noise_min)
    else:
        noise = np.zeros_like(noise)
    
    return (noise * 255).astype(np.uint8)

# ====================================
# CELDA 4: EA1 - Clasificación Visual
# ====================================

LABELS = ["Persona", "Objeto", "Paisaje"]

def vision_classifier(image):
    """Clasificación simulada de imágenes"""
    try:
        if image is None:
            return "⚠️ No se proporcionó imagen"
        
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        
        img = image.resize((128, 128))
        img_array = np.array(img)
        
        brightness = img_array.mean()
        std = img_array.std()
        
        logit_persona = 2.0 + (brightness / 255) * 0.5
        logit_objeto = 1.0 + (std / 128) * 0.3
        logit_paisaje = -0.5 + (brightness / 255) * 0.8
        
        logits = torch.tensor([logit_persona, logit_objeto, logit_paisaje])
        probs = torch.softmax(logits, dim=0).numpy()
        
        resultado = "🔍 Clasificación Visual:\n\n"
        for i, label in enumerate(LABELS):
            resultado += f"{label}: {probs[i]:.2%}\n"
        
        return resultado
    
    except Exception as e:
        return f"❌ Error: {str(e)}"

# ====================================
# CELDA 5: EA2 - Análisis de Sentimientos
# ====================================

def sentiment_analysis(texto):
    """Análisis de sentimientos basado en palabras clave"""
    if not texto or not isinstance(texto, str):
        return "⚠️ Error: Texto inválido"
    
    texto = texto.lower()
    
    positivos = ["bueno", "excelente", "agradable", "feliz", "maravilloso", 
                 "positivo", "genial", "fantástico", "increíble", "hermoso"]
    negativos = ["malo", "terrible", "horrible", "triste", "pésimo", 
                 "negativo", "feo", "desagradable", "molesto", "aburrido"]
    
    score = 0
    palabras_encontradas = []
    
    for p in positivos:
        if p in texto:
            score += 1
            palabras_encontradas.append(f"+{p}")
    
    for n in negativos:
        if n in texto:
            score -= 1
            palabras_encontradas.append(f"-{n}")
    
    if score > 0:
        resultado = "😊 Sentimiento POSITIVO"
    elif score < 0:
        resultado = "😞 Sentimiento NEGATIVO"
    else:
        resultado = "😐 Sentimiento NEUTRAL"
    
    if palabras_encontradas:
        resultado += f"\n\nPalabras detectadas: {', '.join(palabras_encontradas[:3])}"
    
    return resultado

# ====================================
# CELDA 6: EA3 - Generación Sintética
# ====================================

def generar_imagen_sintetica(seed, modo):
    """Genera imagen sintética usando ruido Perlin"""
    try:
        seed = int(seed) if seed is not None else 42
        np.random.seed(seed)
        
        if modo == "Multi-octave":
            noise = np.zeros((256, 256))
            for octave in range(4):
                scale = 8 * (2 ** octave)
                amplitude = 1.0 / (2 ** octave)
                noise += perlin_noise(256, 256, scale=scale) * amplitude
            noise = ((noise - noise.min()) / (noise.max() - noise.min()) * 255).astype(np.uint8)
            
        elif modo == "Colorizado":
            noise = perlin_noise(256, 256, scale=14)
            img_rgb = np.zeros((256, 256, 3), dtype=np.uint8)
            img_rgb[:,:,0] = noise
            img_rgb[:,:,1] = (noise * 0.7).astype(np.uint8)
            img_rgb[:,:,2] = (noise * 0.4).astype(np.uint8)
            return Image.fromarray(img_rgb, mode='RGB')
        
        else:  # Perlin básico
            noise = perlin_noise(256, 256, scale=14)
        
        return Image.fromarray(noise, mode='L')
    
    except Exception as e:
        error_img = Image.new('L', (256, 256), color=128)
        return error_img

# ====================================
# CELDA 7: Pipeline Integrado
# ====================================

def pipeline_integrado(image, text_input, seed, modo_generacion):
    """Pipeline completo multimodal"""
    
    try:
        seed = int(seed) if seed is not None else 42
    except:
        seed = 42
    
    # EA1: Visión
    if image is None:
        salida_vision = "⚠️ Aviso: Sube una imagen para procesar EA1"
    else:
        salida_vision = vision_classifier(image)
    
    # EA2: NLP
    if text_input is None or str(text_input).strip() == "":
        salida_texto = "⚠️ Error: Debes escribir un texto para análisis"
    else:
        salida_texto = sentiment_analysis(text_input)
    
    # EA3: Generación
    salida_sintetica = generar_imagen_sintetica(seed, modo_generacion)
    
    return salida_vision, salida_texto, salida_sintetica

# ====================================
# CELDA 8: Interfaz Gradio
# ====================================

interface = gr.Interface(
    fn=pipeline_integrado,
    inputs=[
        gr.Image(label="📷 Imagen de entrada", type="pil"),
        gr.Textbox(
            label="📝 Texto para análisis", 
            placeholder="Escribe algo... (ej: 'Este es un día maravilloso')"
        ),
        gr.Slider(
            minimum=0,
            maximum=9999,
            value=42,
            step=1,
            label="🎲 Semilla de generación"
        ),
        gr.Radio(
            choices=["Perlin", "Multi-octave", "Colorizado"],
            value="Perlin",
            label="🎨 Modo de generación sintética"
        )
    ],
    outputs=[
        gr.Textbox(label="🔍 Clasificación Visual (EA1)"),
        gr.Textbox(label="💬 Análisis de Sentimiento (EA2)"),
        gr.Image(label="🎨 Imagen Sintética (EA3)")
    ],
    title="🌐 Sistema Integrado Multimodal",
    description="""
    **Visión por Computadora + PLN + Generación Sintética**
    
    - **EA1**: Clasificación de imágenes (Persona/Objeto/Paisaje)
    - **EA2**: Análisis de sentimientos en texto
    - **EA3**: Generación de imágenes sintéticas con ruido Perlin
    
    Prueba con diferentes imágenes, textos y semillas para ver resultados variados.
    """,
    examples=[
        [None, "Me siento muy feliz y positivo hoy", 42, "Perlin"],
        [None, "Esto es terrible y horrible", 123, "Multi-octave"],
        [None, "El día está normal", 999, "Colorizado"]
    ],
    theme="default"
)

interface.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 34.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.18.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 11.0.3 which is incompatible.
google-genai 1.48.0 requires websockets<15.1.0,>=13.0.0, but you have websockets 11.0.3 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which 

In [ ]:
# ====================================
# CELDA 1: Instalación
# ====================================
!pip install -q gradio==3.50.2
!pip install -q transformers torch torchvision
!pip install -q sentencepiece protobuf

# ====================================
# CELDA 2: Importaciones
# ====================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import gradio as gr
import torch
from transformers import pipeline
from torchvision import transforms
import torchvision.models as models
import torch.nn.functional as F

print("✅ Librerías cargadas correctamente")

# ====================================
# CELDA 3: Cargar Modelos de IA
# ====================================

print("🔄 Cargando modelos de IA...")

# Modelo de clasificación de imágenes (MobileNetV2 - ligero y rápido)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Usando dispositivo: {device}")

vision_model = models.mobilenet_v2(pretrained=True)
vision_model.eval()
vision_model = vision_model.to(device)

# Transformaciones para el modelo de visión
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Cargar etiquetas de ImageNet directamente desde URL
import urllib.request
import json

try:
    # Intentar descargar las etiquetas de ImageNet
    url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
    with urllib.request.urlopen(url) as response:
        imagenet_labels = json.loads(response.read().decode())
    print("✅ Etiquetas de ImageNet cargadas")
except:
    # Si falla, usar etiquetas básicas
    imagenet_labels = [f"Categoría {i}" for i in range(1000)]
    print("⚠️ Usando etiquetas genéricas")

# Modelo de análisis de sentimientos (DistilBERT)
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device == "cuda" else -1
)

print("✅ Modelos cargados exitosamente")

# ====================================
# CELDA 4: EA1 - Clasificación Visual REAL
# ====================================

def vision_classifier_real(image):
    """
    Clasificación real usando MobileNetV2 entrenado en ImageNet
    Detecta 1000 categorías diferentes de objetos, animales, personas, etc.
    """
    try:
        if image is None:
            return "⚠️ No se proporcionó imagen"
        
        # Asegurar que es PIL Image
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        
        # Preprocesar imagen
        input_tensor = preprocess(image)
        input_batch = input_tensor.unsqueeze(0).to(device)
        
        # Hacer predicción
        with torch.no_grad():
            output = vision_model(input_batch)
        
        # Obtener probabilidades
        probabilities = F.softmax(output[0], dim=0)
        
        # Top 5 predicciones
        top5_prob, top5_catid = torch.topk(probabilities, 5)
        
        resultado = "🔍 **Clasificación Visual con IA Real** (MobileNetV2)\n\n"
        resultado += "**Top 5 predicciones:**\n\n"
        
        for i in range(5):
            cat_id = top5_catid[i].item()
            label = imagenet_labels[cat_id] if cat_id < len(imagenet_labels) else f"Categoría {cat_id}"
            prob = top5_prob[i].item()
            resultado += f"{i+1}. **{label}**: {prob:.2%}\n"
        
        return resultado
    
    except Exception as e:
        return f"❌ Error en clasificación: {str(e)}"



# ====================================
# CELDA 5: EA2 - Análisis de Sentimientos REAL
# ====================================

def sentiment_analysis_real(texto):
    """
    Análisis de sentimientos usando DistilBERT
    Modelo pre-entrenado en millones de textos para detectar emociones
    """
    if not texto or not isinstance(texto, str) or texto.strip() == "":
        return "⚠️ Error: Debes proporcionar un texto válido"
    
    try:
        # Análisis con el modelo
        result = sentiment_analyzer(texto[:512])  # Limitar a 512 tokens
        
        label = result[0]['label']
        score = result[0]['score']
        
        # Traducir etiquetas
        if label == "POSITIVE":
            emoji = "😊"
            sentimiento = "POSITIVO"
        else:
            emoji = "😞"
            sentimiento = "NEGATIVO"
        
        resultado = f"{emoji} **Sentimiento {sentimiento}**\n\n"
        resultado += f"**Confianza del modelo:** {score:.2%}\n\n"
        resultado += f"**Texto analizado:** \"{texto[:100]}{'...' if len(texto) > 100 else ''}\"\n\n"
        resultado += f"_Modelo: DistilBERT (Transformers)_"
        
        return resultado
    
    except Exception as e:
        return f"❌ Error en análisis: {str(e)}"

# ====================================
# CELDA 6: Generador Perlin Mejorado
# ====================================

def fade(t):
    """Función de suavizado Perlin"""
    return 6*t**5 - 15*t**4 + 10*t**3

def perlin_noise(width, height, scale=12):
    """Genera ruido Perlin de alta calidad"""
    grid_w = width // scale + 2
    grid_h = height // scale + 2

    gradients = np.random.randn(grid_h, grid_w, 2)
    norms = np.linalg.norm(gradients, axis=2, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    gradients = gradients / norms

    xs = np.linspace(0, grid_w - 2, width)
    ys = np.linspace(0, grid_h - 2, height)
    x, y = np.meshgrid(xs, ys)

    x0 = np.clip(x.astype(int), 0, grid_w - 2)
    x1 = np.clip(x0 + 1, 0, grid_w - 1)
    y0 = np.clip(y.astype(int), 0, grid_h - 2)
    y1 = np.clip(y0 + 1, 0, grid_h - 1)

    dx = x - x0
    dy = y - y0

    g00 = gradients[y0, x0]
    g10 = gradients[y0, x1]
    g01 = gradients[y1, x0]
    g11 = gradients[y1, x1]

    dot00 = g00[:,:,0]*dx + g00[:,:,1]*dy
    dot10 = g10[:,:,0]*(dx-1) + g10[:,:,1]*dy
    dot01 = g01[:,:,0]*dx + g01[:,:,1]*(dy-1)
    dot11 = g11[:,:,0]*(dx-1) + g11[:,:,1]*(dy-1)

    u = fade(dx)
    v = fade(dy)

    noise = (1-u)*(1-v)*dot00 + u*(1-v)*dot10 + (1-u)*v*dot01 + u*v*dot11

    noise_min = noise.min()
    noise_max = noise.max()
    if noise_max - noise_min > 0:
        noise = (noise - noise_min) / (noise_max - noise_min)
    else:
        noise = np.zeros_like(noise)
    
    return noise

# ====================================
# CELDA 7: EA3 - Generación Sintética Avanzada
# ====================================

def generar_imagen_sintetica_avanzada(seed, modo):
    """
    Generador de imágenes sintéticas de alta calidad
    Usa múltiples técnicas procedurales para crear arte generativo
    """
    try:
        seed = int(seed) if seed is not None else 42
        np.random.seed(seed)
        
        if modo == "Nubes Realistas":
            # Perlin multi-octava para nubes
            noise = np.zeros((512, 512))
            for octave in range(6):
                scale = 8 * (2 ** octave)
                amplitude = 1.0 / (2 ** octave)
                noise += perlin_noise(512, 512, scale=scale) * amplitude
            
            # Normalizar
            noise = (noise - noise.min()) / (noise.max() - noise.min())
            
            # Aplicar curva para hacer nubes más dramáticas
            noise = np.power(noise, 1.5)
            noise = (noise * 255).astype(np.uint8)
            
            img = Image.fromarray(noise, mode='L')
            img = img.filter(ImageFilter.GaussianBlur(radius=1))
            
            return img
            
        elif modo == "Mármol Procedural":
            # Patrón de mármol usando Perlin
            noise1 = perlin_noise(512, 512, scale=20)
            noise2 = perlin_noise(512, 512, scale=10)
            
            # Combinar capas
            marble = noise1 * 0.7 + noise2 * 0.3
            
            # Agregar vetas
            x = np.linspace(0, 10, 512)
            y = np.linspace(0, 10, 512)
            X, Y = np.meshgrid(x, y)
            veins = np.sin(X + marble * 5) * 0.3
            
            marble = marble + veins
            marble = (marble - marble.min()) / (marble.max() - marble.min())
            marble = (marble * 255).astype(np.uint8)
            
            img = Image.fromarray(marble, mode='L')
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(1.5)
            
            return img
            
        elif modo == "Paisaje Alienígena":
            # Generar terreno con colores surrealistas
            terrain = np.zeros((512, 512))
            for octave in range(5):
                scale = 12 * (2 ** octave)
                amplitude = 1.0 / (1.5 ** octave)
                terrain += perlin_noise(512, 512, scale=scale) * amplitude
            
            terrain = (terrain - terrain.min()) / (terrain.max() - terrain.min())
            
            # Crear imagen RGB con colores alienígenas
            img_rgb = np.zeros((512, 512, 3), dtype=np.uint8)
            
            # Canal rojo: púrpuras
            img_rgb[:,:,0] = (terrain * 180 + 75).astype(np.uint8)
            # Canal verde: verdes cyan
            img_rgb[:,:,1] = (np.power(terrain, 0.7) * 200 + 55).astype(np.uint8)
            # Canal azul: azules profundos
            img_rgb[:,:,2] = (np.power(terrain, 1.3) * 220 + 35).astype(np.uint8)
            
            img = Image.fromarray(img_rgb, mode='RGB')
            img = img.filter(ImageFilter.SMOOTH)
            
            return img
            
        elif modo == "Textura Orgánica":
            # Múltiples capas de Perlin con diferentes escalas
            base = perlin_noise(512, 512, scale=15)
            detail = perlin_noise(512, 512, scale=5)
            fine = perlin_noise(512, 512, scale=2)
            
            combined = base * 0.5 + detail * 0.3 + fine * 0.2
            combined = (combined - combined.min()) / (combined.max() - combined.min())
            
            # Crear gradiente de color
            img_rgb = np.zeros((512, 512, 3), dtype=np.uint8)
            img_rgb[:,:,0] = (combined * 200 + 55).astype(np.uint8)
            img_rgb[:,:,1] = (np.power(combined, 0.8) * 180 + 75).astype(np.uint8)
            img_rgb[:,:,2] = (np.power(combined, 1.2) * 160 + 95).astype(np.uint8)
            
            img = Image.fromarray(img_rgb, mode='RGB')
            enhancer = ImageEnhance.Sharpness(img)
            img = enhancer.enhance(1.3)
            
            return img
        
        else:  # Perlin Clásico
            noise = perlin_noise(512, 512, scale=18)
            noise = (noise * 255).astype(np.uint8)
            return Image.fromarray(noise, mode='L')
    
    except Exception as e:
        print(f"Error en generación: {e}")
        error_img = Image.new('RGB', (512, 512), color=(100, 100, 100))
        return error_img

# ====================================
# CELDA 8: Pipeline Integrado
# ====================================

def pipeline_multimodal(image, text_input, seed, modo_generacion):
    """
    Pipeline completo con IA real en las tres modalidades
    """
    
    try:
        seed = int(seed) if seed is not None else 42
    except:
        seed = 42
    
    # EA1: Clasificación de Imágenes con IA
    if image is None:
        salida_vision = "⚠️ **Aviso:** Sube una imagen para clasificarla con IA\n\nEl sistema usa MobileNetV2 entrenado en ImageNet con 1000 categorías."
    else:
        salida_vision = vision_classifier_real(image)
    
    # EA2: Análisis de Sentimientos con IA
    if text_input is None or str(text_input).strip() == "":
        salida_texto = "⚠️ **Error:** Proporciona un texto para análisis\n\nEl sistema usa DistilBERT, un modelo transformer de última generación."
    else:
        salida_texto = sentiment_analysis_real(text_input)
    
    # EA3: Generación Sintética Avanzada
    salida_sintetica = generar_imagen_sintetica_avanzada(seed, modo_generacion)
    
    return salida_vision, salida_texto, salida_sintetica

# ====================================
# CELDA 9: Interfaz Gradio
# ====================================

interface = gr.Interface(
    fn=pipeline_multimodal,
    inputs=[
        gr.Image(label="📷 Imagen para Clasificación", type="pil"),
        gr.Textbox(
            label="📝 Texto para Análisis de Sentimientos", 
            placeholder="Ejemplo: 'I love this product! It's amazing and works perfectly.'",
            lines=3
        ),
        gr.Slider(
            minimum=0,
            maximum=9999,
            value=42,
            step=1,
            label="🎲 Semilla para Generación (cambia para diferentes resultados)"
        ),
        gr.Radio(
            choices=[
                "Perlin Clásico",
                "Nubes Realistas", 
                "Mármol Procedural",
                "Paisaje Alienígena",
                "Textura Orgánica"
            ],
            value="Nubes Realistas",
            label="🎨 Estilo de Imagen Sintética"
        )
    ],
    outputs=[
        gr.Textbox(label="🔍 EA1: Clasificación Visual (MobileNetV2)", lines=8),
        gr.Textbox(label="💬 EA2: Análisis de Sentimientos (DistilBERT)", lines=6),
        gr.Image(label="🎨 EA3: Imagen Sintética Generada (512x512)", type="pil")
    ],
    title="🤖 Sistema Multimodal con IA Real",
    description="""
    ## Sistema Integrado de Inteligencia Artificial Multimodal
    
    ### 🎯 Capacidades:
    
    - **EA1 - Visión por Computadora:** Usa **MobileNetV2** entrenado en ImageNet (1.4M imágenes, 1000 categorías)
    - **EA2 - Procesamiento de Lenguaje:** Usa **DistilBERT** (modelo transformer) para análisis emocional
    - **EA3 - Arte Generativo:** Algoritmos procedurales avanzados (Perlin multi-octava, patrones orgánicos)
    
    ### 💡 Cómo usar:
    1. Sube una imagen (animales, objetos, paisajes, personas)
    2. Escribe un texto en inglés para analizar su sentimiento
    3. Ajusta la semilla para generar diferentes imágenes sintéticas
    4. Selecciona el estilo de arte generativo que prefieras
    
    _Todos los modelos se ejecutan localmente. La primera ejecución puede tardar mientras se descargan los modelos._
    """,
    examples=[
        [None, "This is absolutely wonderful! I'm so happy with the results.", 42, "Nubes Realistas"],
        [None, "I hate this. It's terrible and disappointing.", 123, "Mármol Procedural"],
        [None, "The product is okay, nothing special really.", 999, "Paisaje Alienígena"]
    ],
    theme="default",
    allow_flagging="never"
)

print("\n🚀 Lanzando interfaz...\n")
interface.launch(share=True, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 67.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires rmm-cu12==25.6.*, but you have rmm-cu12 25.2.0 which is incompatible.


2025-12-06 03:10:44.958670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764990645.137348      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764990645.191120      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


✅ Modelos cargados exitosamente

🚀 Lanzando interfaz...

Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://9c2ae1c416454c6462.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
